In [1]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from useful_functions import *
import pandas as pd
from joblib import Parallel, delayed

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
color_data      = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_with_Flux.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

ids = read_ids('parent_samples_ids.txt')

SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=ids)


In [3]:
# SPECTRA = SPECTRA.subtype_filter(subtype='QSO', exclude=True)
SPECTRA.stack_data()
# # SPECTRA.mask_bad()

In [4]:
FIT     = FitSpectrum()
SPECTRA = FIT.shift_to_rest_frame(SPECTRA)
SPECTRA = FIT.label_emission_lines(SPECTRA, 3)
print(SPECTRA.n_spectra)


61904


In [5]:
# SPECTRA = FIT.significant_emission_filter(SPECTRA)
# print(SPECTRA.n_spectra)

In [6]:
# with open('parent_samples_ids.txt', 'w+') as f:
#     for tid in SPECTRA.df['TARGETID'].to_list():
#         f.write(f'{tid}\n')

In [7]:
SPECTRA.df.head(5)

,TARGETID,RA,DEC,SPECTYPE,LOGM,LOGSFR,z_pipe,z,OII,Hbeta,OIII,Halpha,NII,SII
0,39627739380056197,177.003822,-1.903397,GALAXY,10.250155,-0.348563,0.224113,0.224113,False,False,False,True,True,False
1,39627739380056420,177.013472,-1.972766,GALAXY,9.722920,-0.349153,0.102005,0.102005,False,False,False,True,False,False
2,39627739380056564,177.019312,-1.965308,GALAXY,10.452706,0.941032,0.323187,0.323187,True,False,False,True,True,False
3,39627739380056768,177.028143,-1.960056,GALAXY,10.143390,0.193251,0.102814,0.102814,True,True,False,True,False,True
4,39627739380056864,177.032977,-1.909970,GALAXY,10.575973,-9.632146,0.167324,0.167324,False,False,False,True,True,False


In [8]:
def process_target(target_id):
    """
    Processes a single target to find double-peaked features.
    """
    # try:
    p_value, delta_dv, dp_detection, line_fluxes_rank, params_2comp = FIT.find_dp(SPECTRA, id=target_id)

    dp_cols = ['OII3726_dp', 'OII3729_dp',
            'Hbeta_dp',
            'OIII4959_dp', 'OIII5007_dp',
            'NII6548_dp', 'Halpha_dp', 'NII6583_dp', 
            'SII6716_dp', 'SII6731_dp']

    dp_rank_cols = [f'{col[:-3]}_rank' for col in dp_cols]

    data = {
        'TARGETID': target_id,
        'p_value': p_value,
        'dv_r': params_2comp['dv'][0],
        'dv_l': params_2comp['dv'][1],
        'delta_dv': delta_dv,
        'sigma_r': params_2comp['sigma'][0],
        'sigma_l': params_2comp['sigma'][1],
    }
    data.update(dict(zip(dp_cols, dp_detection)))
    data.update(dict(zip(dp_rank_cols, line_fluxes_rank)))
    return data
    # except Exception as e:
    #     print(f"Error processing target {target_id}: {e}")
    #     pass

    

# Use joblib to parallelize the processing over all target IDs
# n_jobs=-1 uses all available CPU cores.
results = Parallel(n_jobs=10)(delayed(process_target)(target_id) for target_id in tqdm(SPECTRA.targetID))

# Convert the list of dictionaries to a DataFrame
dp_df = pd.DataFrame(results)

# display(dp_df)

100%|██████████| 61904/61904 [05:32<00:00, 186.02it/s]


In [9]:
display(dp_df.head(10))

,TARGETID,p_value,dv_r,dv_l,delta_dv,sigma_r,sigma_l,OII3726_dp,OII3729_dp,Hbeta_dp,...,OII3726_rank,OII3729_rank,Hbeta_rank,OIII4959_rank,OIII5007_rank,NII6548_rank,Halpha_rank,NII6583_rank,SII6716_rank,SII6731_rank
0,39627739380056197,0.670855,15.713979,-83.749776,99.463755,43.278020,28.635134,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
1,39627739380056420,0.206380,44.028876,-21.983614,66.012490,0.010000,32.578532,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
2,39627739380056564,0.000047,54.235580,-77.214870,131.450450,50.133639,31.151569,False,False,False,...,3,1,-1,-1,-1,4,0,2,-1,-1
3,39627739380056768,0.054544,3.450798,-131.142945,134.593743,44.644933,0.010000,False,False,False,...,3,1,4,-1,-1,7,0,2,5,6
4,39627739380056864,0.348877,4.887532,-181.647119,186.534652,86.723750,53.832209,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
5,39627739380057310,0.664107,7.579483,-13.979119,21.558602,32.881477,0.010000,False,False,False,...,2,1,-1,7,3,8,0,6,5,4
6,39627739380057506,0.000199,120.705683,-57.385546,178.091228,39.709895,73.005164,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
7,39627739380057763,0.000162,9.268286,-51.850184,61.118471,64.180774,121.967433,False,False,False,...,-1,-1,3,-1,-1,2,0,1,-1,-1
8,39627739380057802,0.829053,67.906590,-25.782736,93.689325,0.010063,190.088922,False,False,False,...,1,0,-1,-1,-1,-1,-1,-1,-1,-1
9,39627739380058028,0.035712,15.950163,-6.314953,22.265116,0.010000,37.822797,False,False,True,...,-1,-1,2,-1,-1,4,0,1,3,5


In [10]:
dp_df.to_csv('dp_parent_results.csv', index=False)

In [11]:
# dp_df = pd.read_csv('dp_parent_results.csv', low_memory=False)
print(f'All: {len(dp_df)}')
dp_candidates = dp_df[(dp_df['p_value'] < 0.05) & (dp_df['delta_dv'] > 75)]
print(f'DP Candidates: {len(dp_candidates)} ({len(dp_candidates) / len(dp_df) * 100:.2f}%)')

All: 61904
DP Candidates: 26018 (42.03%)
